# gis inclusion survey analysis - regrouped

this notebook uses the cleaned survey file and keeps all real entries.

important setup:
- june = before the curriculum changes
- december = after the curriculum changes
- i am ignoring the alien question and not using it in the analysis
- the goal is to see the actual pattern, not force the after results to look better


In [ ]:
# packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from pathlib import Path

# file paths
DATA_PATH = Path('/mnt/data/gis_survey_cleaned.xlsx')
OUT_DIR = Path('/mnt/data/gis_regrouped_outputs')
OUT_DIR.mkdir(exist_ok=True)

# so i can see more columns when checking tables
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 80)


## load the survey

first i load the cleaned excel file. i also remove the repeated qualtrics header row if it is still there.


In [ ]:
# load the data
# change the sheet name here if your excel sheet name changes later
try:
    df_raw = pd.read_excel(DATA_PATH, sheet_name='Snigdha Copy of GIS survey_clea')
except ValueError:
    df_raw = pd.read_excel(DATA_PATH)

# make a copy so the original stays untouched
df = df_raw.copy()

# this removes the repeated header row because it will not parse as a real date
df['StartDate_parsed'] = pd.to_datetime(df['StartDate'], errors='coerce')
df = df[df['StartDate_parsed'].notna()].copy()

# june = before, december = after
df['Period'] = np.where(
    df['StartDate_parsed'].dt.month == 6, 'Before',
    np.where(df['StartDate_parsed'].dt.month == 12, 'After', 'Other')
)

# keep only the two survey waves
df = df[df['Period'].isin(['Before', 'After'])].copy()
df['Period'] = pd.Categorical(df['Period'], categories=['Before', 'After'], ordered=True)

print('usable responses:', len(df))
print(df['Period'].value_counts().sort_index())


## recode the survey answers

here i turn the text answers into numbers. higher numbers mean a more positive response.

i am keeping the activity questions separate because they use a slightly different scale.


In [ ]:
# main scale used by most representation/case study/material questions
main_scale = {
    'No': 1,
    'Very less': 1,
    'Less': 2,
    'Maybe': 3,
    'Moderate': 3,
    'Very well': 4
}

# scale used by theory/demo/homework/final project questions
activity_scale = {
    'Not well at all': 1,
    'Slightly well': 2,
    'Moderately well': 3,
    'Very well': 4,
    'Extremely well': 5,
    'Extreamly well': 5  # keeping this in case the spelling exists in the data
}

# these are the main questions i am using
main_items = [
    'Q2.2','Q2.3','Q2.4','Q2.5','Q2.7',
    'Q3.2','Q3.3','Q3.5','Q3.6','Q3.7',
    'Q4.2','Q4.3','Q4.4','Q4.5','Q5.7'
]

activity_items = ['Q5.3','Q5.4','Q5.5','Q5.6']

# convert the responses to numbers
for col in main_items:
    df[col + '_num'] = df[col].map(main_scale)

for col in activity_items:
    df[col + '_num'] = df[col].map(activity_scale)

# quick check for answers that did not get mapped
for col in main_items + activity_items:
    unmapped = df.loc[df[col].notna() & df[col + '_num'].isna(), col].unique()
    if len(unmapped) > 0:
        print('unmapped values in', col, ':', unmapped)


## regroup the questions

instead of one big inclusivity index, i am separating the questions into smaller groups.

this should make the results easier to explain, especially because the before group is small and looks very positive in some places.


In [ ]:
# tighter constructs for the paper
constructs = {
    'Overall curriculum representation': [
        'Q2.2_num', # diverse perspectives
        'Q2.3_num', # gender diversity
        'Q2.4_num', # diverse demographics
        'Q2.5_num'  # geographic regions
    ],

    'Responsible spatial practice': [
        'Q2.7_num', # understanding social/cultural issues
        'Q3.7_num', # social/cultural issues in case studies
        'Q5.7_num'  # connecting GIS skills to societal problems
    ],

    'Geographic inclusion': [
        'Q2.5_num', # geographic regions in curriculum
        'Q3.5_num', # geographic regions in case studies
        'Q4.4_num'  # geographic regions in materials
    ],

    'Demographic and gender inclusion': [
        'Q2.3_num', # gender diversity in curriculum
        'Q2.4_num', # diverse demographics in curriculum
        'Q3.3_num', # gender diversity in case studies
        'Q3.6_num', # diverse demographics in case studies
        'Q4.3_num', # gender diversity in materials
        'Q4.5_num'  # diverse demographics in materials
    ],

    'Course artifacts': [
        'Q3.2_num', # diverse case studies
        'Q4.2_num'  # diverse materials
    ],

    'Teaching activities': [
        'Q5.3_num', # theoretical lectures
        'Q5.4_num', # in-class demos
        'Q5.5_num', # homework
        'Q5.6_num'  # final projects
    ]
}

# calculate one score for each construct
for name, cols in constructs.items():
    df[name] = df[cols].mean(axis=1)

construct_cols = list(constructs.keys())

df[['Period'] + construct_cols].head()


## reliability check

cronbach's alpha is a quick check for whether the questions inside each group behave like they measure the same thing.

because the sample is small, i am using this as a diagnostic and not as a strict rule.


In [ ]:
def cronbach_alpha(data):
    # remove rows with missing values for this construct
    data = data.dropna(axis=0, how='any')

    # alpha needs at least two items and two responses
    if data.shape[1] < 2 or data.shape[0] < 2:
        return np.nan

    item_vars = data.var(axis=0, ddof=1)
    total_var = data.sum(axis=1).var(ddof=1)
    n_items = data.shape[1]

    if total_var == 0:
        return np.nan

    return (n_items / (n_items - 1)) * (1 - item_vars.sum() / total_var)

alpha_rows = []
for name, cols in constructs.items():
    alpha_rows.append({
        'Construct': name,
        'Items': ', '.join([c.replace('_num','') for c in cols]),
        'Number of items': len(cols),
        'Cronbach alpha': round(cronbach_alpha(df[cols]), 3)
    })

alpha_table = pd.DataFrame(alpha_rows)
alpha_table


## compare before and after

this is the main table for the paper. it compares june and december for each construct.

since these are not the same students, i am using mann-whitney u tests instead of paired tests.


In [ ]:
def cohens_d(before, after):
    before = pd.Series(before).dropna()
    after = pd.Series(after).dropna()

    if len(before) < 2 or len(after) < 2:
        return np.nan

    pooled_sd = np.sqrt(
        ((len(before) - 1) * before.var(ddof=1) + (len(after) - 1) * after.var(ddof=1)) /
        (len(before) + len(after) - 2)
    )

    if pooled_sd == 0:
        return np.nan

    return (after.mean() - before.mean()) / pooled_sd

comparison_rows = []

for col in construct_cols:
    before = df.loc[df['Period'] == 'Before', col].dropna()
    after = df.loc[df['Period'] == 'After', col].dropna()

    if len(before) > 0 and len(after) > 0:
        u_stat, p_value = mannwhitneyu(before, after, alternative='two-sided')
    else:
        u_stat, p_value = np.nan, np.nan

    comparison_rows.append({
        'Construct': col,
        'Before n': len(before),
        'After n': len(after),
        'Before mean': round(before.mean(), 3),
        'After mean': round(after.mean(), 3),
        'Difference: After - Before': round(after.mean() - before.mean(), 3),
        'Percent change': round(((after.mean() - before.mean()) / before.mean()) * 100, 1) if before.mean() != 0 else np.nan,
        'Mann-Whitney p-value': round(p_value, 4) if pd.notna(p_value) else np.nan,
        "Cohen's d": round(cohens_d(before, after), 3)
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table


In [ ]:
# bar chart of construct means
plot_data = comparison_table.set_index('Construct')[['Before mean', 'After mean']]

ax = plot_data.plot(kind='bar', figsize=(10, 5), rot=35)
ax.set_title('before vs after mean scores')
ax.set_ylabel('mean score')
ax.set_xlabel('')
ax.legend(['before - june', 'after - december'])

plt.tight_layout()
plt.savefig(OUT_DIR / 'construct_means_before_after.png', dpi=300)
plt.show()


## favorable response share

means can be misleading when the before group is small. so i am also checking the share of clearly positive answers.

for the 1-4 questions, favorable means `very well`.
for the 1-5 activity questions, favorable means `very well` or `extremely well`.


In [ ]:
# make favorable-response columns
for col in main_items:
    df[col + '_fav'] = df[col + '_num'].eq(4)

for col in activity_items:
    df[col + '_fav'] = df[col + '_num'].ge(4)

# calculate favorable share for each construct
for name, cols in constructs.items():
    fav_cols = [c.replace('_num', '_fav') for c in cols]
    df[name + ' - favorable share'] = df[fav_cols].mean(axis=1)

fav_rows = []

for name in constructs:
    col = name + ' - favorable share'
    before = df.loc[df['Period'] == 'Before', col].dropna()
    after = df.loc[df['Period'] == 'After', col].dropna()

    fav_rows.append({
        'Construct': name,
        'Before favorable %': round(before.mean() * 100, 1),
        'After favorable %': round(after.mean() * 100, 1),
        'Difference percentage points': round((after.mean() - before.mean()) * 100, 1)
    })

favorable_table = pd.DataFrame(fav_rows)
favorable_table


In [ ]:
# plot favorable responses
plot_data = favorable_table.set_index('Construct')[['Before favorable %', 'After favorable %']]

ax = plot_data.plot(kind='bar', figsize=(10, 5), rot=35)
ax.set_title('before vs after favorable response share')
ax.set_ylabel('favorable responses (%)')
ax.set_xlabel('')
ax.legend(['before - june', 'after - december'])

plt.tight_layout()
plt.savefig(OUT_DIR / 'favorable_share_before_after.png', dpi=300)
plt.show()


## question-level check

this helps me see which exact questions improved or declined. this is useful because a broad index can hide the actual pattern.


In [ ]:
question_labels = {
    'Q2.2_num': 'curriculum represents diverse perspectives',
    'Q2.3_num': 'curriculum represents gender diversity',
    'Q2.4_num': 'curriculum represents diverse demographics',
    'Q2.5_num': 'curriculum represents geographic regions',
    'Q2.7_num': 'course builds understanding of social/cultural issues',
    'Q3.2_num': 'case studies from diverse groups',
    'Q3.3_num': 'case studies on gender diversity',
    'Q3.5_num': 'case studies on diverse regions',
    'Q3.6_num': 'case studies on diverse demographics',
    'Q3.7_num': 'case studies on social/cultural issues',
    'Q4.2_num': 'materials on diverse groups',
    'Q4.3_num': 'materials on gender diversity',
    'Q4.4_num': 'materials on diverse regions',
    'Q4.5_num': 'materials on diverse demographics',
    'Q5.3_num': 'diverse perspectives in theoretical lectures',
    'Q5.4_num': 'diverse perspectives in demo exercises',
    'Q5.5_num': 'diverse perspectives in homework',
    'Q5.6_num': 'diverse perspectives in final projects',
    'Q5.7_num': 'connect GIS skills to societal problems'
}

question_rows = []

for col, label in question_labels.items():
    before = df.loc[df['Period'] == 'Before', col].dropna()
    after = df.loc[df['Period'] == 'After', col].dropna()

    question_rows.append({
        'Question': col.replace('_num', ''),
        'Label': label,
        'Before mean': round(before.mean(), 3),
        'After mean': round(after.mean(), 3),
        'Difference': round(after.mean() - before.mean(), 3),
        'Before n': len(before),
        'After n': len(after)
    })

question_level_table = pd.DataFrame(question_rows)
question_level_table.sort_values('Difference', ascending=False)


In [ ]:
# plot question-level differences
plot_df = question_level_table.sort_values('Difference')

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(plot_df['Label'], plot_df['Difference'])
ax.axvline(0, linewidth=1)
ax.set_title('question-level change: after minus before')
ax.set_xlabel('mean difference')

plt.tight_layout()
plt.savefig(OUT_DIR / 'question_level_differences.png', dpi=300)
plt.show()


## technical learning

these are not inclusivity questions, so i am keeping them separate.

this helps show whether students still felt technically prepared after the course.


In [ ]:
# summarize GIS background before and after taking the course
tech_before = pd.crosstab(df['Period'], df['Q1.7'], normalize='index').round(3) * 100
tech_after = pd.crosstab(df['Period'], df['Q1.8'], normalize='index').round(3) * 100

print('GIS background before taking the course (%)')
display(tech_before)

print('GIS background after taking the course (%)')
display(tech_after)


## participant characteristics

this is table 1 material for the paper.


In [ ]:
demo_tables = {}

for col in ['Q1.3', 'Q1.4', 'Q1.5', 'Q1.6']:
    demo_tables[col] = pd.crosstab(df['Period'], df[col], margins=True)
    print('
', col)
    display(demo_tables[col])


## open-ended responses

this is just a first pass using keywords. i should still read the responses manually before using the counts in the paper.


In [ ]:
open_cols = {
    'Q6.2': 'what is lacking?',
    'Q6.3': 'underrepresented regions/communities',
    'Q6.4': 'recommended resources/materials'
}

theme_keywords = {
    'Global South / international': ['global south', 'india', 'south asia', 'africa', 'latin america', 'international', 'developing', 'asia'],
    'Indigenous / Native communities': ['indigenous', 'native', 'tribal', 'reservation'],
    'Rural / non-urban': ['rural', 'small town', 'non urban', 'non-urban'],
    'Accessibility / disability': ['accessibility', 'disabled', 'disability', 'ada'],
    'Gender': ['gender', 'women', 'female'],
    'Race / ethnicity': ['race', 'racial', 'black', 'latino', 'hispanic', 'ethnic'],
    'Environmental justice / climate': ['environmental justice', 'climate', 'flood', 'pollution', 'environment'],
    'Housing / income': ['housing', 'income', 'poverty', 'low-income', 'affordability'],
    'Transportation / mobility': ['transport', 'transit', 'mobility', 'access'],
    'Datasets / data sources': ['dataset', 'data source', 'open data', 'census', 'osm', 'openstreetmap']
}

coded_rows = []

for q, q_label in open_cols.items():
    for _, row in df[['Period', q]].dropna().iterrows():
        text = str(row[q]).lower()

        for theme, keywords in theme_keywords.items():
            if any(keyword in text for keyword in keywords):
                coded_rows.append({
                    'Period': row['Period'],
                    'Question': q,
                    'Question label': q_label,
                    'Theme': theme,
                    'Response': row[q]
                })

coded_open = pd.DataFrame(coded_rows)

if not coded_open.empty:
    theme_counts = coded_open.groupby(['Question label', 'Theme']).size().reset_index(name='Count')
else:
    theme_counts = pd.DataFrame(columns=['Question label', 'Theme', 'Count'])

theme_counts


In [ ]:
# plot all coded themes together
if not coded_open.empty:
    combined_theme_counts = coded_open.groupby('Theme').size().sort_values()

    ax = combined_theme_counts.plot(kind='barh', figsize=(8, 5))
    ax.set_title('open-ended response themes')
    ax.set_xlabel('number of coded mentions')
    ax.set_ylabel('')

    plt.tight_layout()
    plt.savefig(OUT_DIR / 'open_ended_theme_counts.png', dpi=300)
    plt.show()
else:
    print('no open-ended themes coded')


## export the tables

this saves the main tables and coded responses so i can use them in the paper and figures later.


In [ ]:
output_xlsx = OUT_DIR / 'regrouped_survey_analysis_tables.xlsx'

with pd.ExcelWriter(output_xlsx, engine='xlsxwriter') as writer:
    df.to_excel(writer, sheet_name='analysis_ready_data', index=False)
    alpha_table.to_excel(writer, sheet_name='reliability_alpha', index=False)
    comparison_table.to_excel(writer, sheet_name='construct_mean_comparison', index=False)
    favorable_table.to_excel(writer, sheet_name='favorable_response_share', index=False)
    question_level_table.to_excel(writer, sheet_name='question_level_differences', index=False)
    tech_before.to_excel(writer, sheet_name='tech_before_distribution')
    tech_after.to_excel(writer, sheet_name='tech_after_distribution')
    theme_counts.to_excel(writer, sheet_name='open_ended_theme_counts', index=False)
    coded_open.to_excel(writer, sheet_name='coded_open_responses', index=False)

print('saved tables to:', output_xlsx)
print('saved figures to:', OUT_DIR)


## how i would write this up

if before still looks better in some places, i should not hide that. i can explain it as:

- the june baseline is small and may have unusually positive respondents
- the june and december groups are not the same students
- the survey is only one part of the evidence
- the stronger evidence should come from the curriculum artifact inventory, syllabus audit, and assignment changes

so the honest claim is not "everything improved." the stronger claim is:

> after the redesign, student perceptions were mixed across broad indices, but the survey helps identify which parts of the curriculum students noticed most clearly. the curriculum audit and artifact analysis are needed to show the actual structural changes in the course.
